In [1]:
import re
import random
from collections import Counter, defaultdict

# ==========================
# Load Corpus
# ==========================
with open("/Users/mahamatsilebo/Downloads/research_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()


In [2]:


# ======================================================
# Sentence Tokenization
# ======================================================

sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
print(sentences)


['\ufeffartificial intelligence has become one of the fastest growing areas of computer science', 'researchers across the world are developing intelligent systems that can perform tasks traditionally requiring human intelligence', 'these tasks include language understanding, image recognition, medical diagnosis, autonomous navigation, financial forecasting, scientific discovery, and decision support', 'machine learning is considered the foundation of modern artificial intelligence', 'instead of relying on manually written rules, machine learning algorithms learn patterns directly from data', 'the quality of the learned model depends on the quality of the training data, feature engineering techniques, algorithm selection, parameter tuning, and evaluation methodology', 'supervised learning uses labelled examples to train predictive models', 'classification algorithms predict discrete class labels such as spam detection or disease diagnosis, while regression algorithms predict continuous 

In [3]:

# ======================================================
# Word Tokenization
# ======================================================

def tokenize(sentence):
    return re.findall(r'\b[a-z]+\b', sentence)

sentence_tokens = [tokenize(sentence) for sentence in sentences]

tokens = []

for sentence in sentence_tokens:
    tokens.extend(sentence)


In [4]:

# ======================================================
# TASK 1
# ======================================================

print("="*60)
print("TASK 1 : CORPUS STATISTICS")
print("="*60)

print("Total Sentences :", len(sentences))
print("Total Words     :", len(tokens))
print("Vocabulary Size :", len(set(tokens)))


TASK 1 : CORPUS STATISTICS
Total Sentences : 42
Total Words     : 639
Vocabulary Size : 394


In [5]:

# ======================================================
# TASK 2
# Generate Unigram, Bigram and Trigram
# ======================================================

unigrams = Counter(tokens)

bigrams = Counter()

trigrams = Counter()

for words in sentence_tokens:

    for i in range(len(words)-1):
        bigrams[(words[i], words[i+1])] += 1

    for i in range(len(words)-2):
        trigrams[(words[i], words[i+1], words[i+2])] += 1

print("\n" + "="*60)
print("TOP 20 UNIGRAMS")
print("="*60)

for word, count in unigrams.most_common(20):
    print(f"{word:20} {count}")

print("\n" + "="*60)
print("TOP 20 BIGRAMS")
print("="*60)

for pair, count in bigrams.most_common(20):
    print(pair, ":", count)

print("\n" + "="*60)
print("TOP 20 TRIGRAMS")
print("="*60)

for tri, count in trigrams.most_common(20):
    print(tri, ":", count)



TOP 20 UNIGRAMS
the                  20
and                  17
of                   15
language             14
word                 10
models               9
words                9
learning             8
intelligence         7
that                 7
to                   7
artificial           6
on                   6
data                 5
processing           5
a                    5
text                 5
machine              4
is                   4
algorithms           4

TOP 20 BIGRAMS
('artificial', 'intelligence') : 6
('of', 'the') : 5
('such', 'as') : 4
('natural', 'language') : 4
('language', 'processing') : 4
('language', 'models') : 4
('machine', 'learning') : 3
('the', 'quality') : 3
('quality', 'of') : 3
('intelligence', 'has') : 2
('has', 'become') : 2
('intelligent', 'systems') : 2
('language', 'understanding') : 2
('on', 'the') : 2
('the', 'training') : 2
('algorithms', 'predict') : 2
('human', 'language') : 2
('models', 'and') : 2
('improves', 'the') : 2
('words', 't

In [6]:

# ======================================================
# TASK 3
# MLE Probabilities
# ======================================================

total_words = len(tokens)

bigram_probability = {}

for (w1, w2), count in bigrams.items():
    bigram_probability[(w1, w2)] = count / unigrams[w1]

trigram_probability = {}

for (w1, w2, w3), count in trigrams.items():
    trigram_probability[(w1, w2, w3)] = count / bigrams[(w1, w2)]

print("\n" + "="*60)
print("TOP 20 BIGRAMS (Highest Probability)")
print("="*60)

top_bigram = sorted(
    bigram_probability.items(),
    key=lambda x: x[1],
    reverse=True
)

for bg, prob in top_bigram[:20]:
    print(bg, ":", round(prob,4))

print("\n" + "="*60)
print("TOP 20 TRIGRAMS (Highest Probability)")
print("="*60)

top_trigram = sorted(
    trigram_probability.items(),
    key=lambda x: x[1],
    reverse=True
)

for tg, prob in top_trigram[:20]:
    print(tg, ":", round(prob,4))



TOP 20 BIGRAMS (Highest Probability)
('artificial', 'intelligence') : 1.0
('has', 'become') : 1.0
('fastest', 'growing') : 1.0
('growing', 'areas') : 1.0
('areas', 'of') : 1.0
('computer', 'science') : 1.0
('across', 'the') : 1.0
('world', 'are') : 1.0
('developing', 'intelligent') : 1.0
('perform', 'tasks') : 1.0
('traditionally', 'requiring') : 1.0
('requiring', 'human') : 1.0
('include', 'language') : 1.0
('image', 'recognition') : 1.0
('medical', 'diagnosis') : 1.0
('autonomous', 'navigation') : 1.0
('navigation', 'financial') : 1.0
('financial', 'forecasting') : 1.0
('forecasting', 'scientific') : 1.0
('discovery', 'and') : 1.0

TOP 20 TRIGRAMS (Highest Probability)
('intelligence', 'has', 'become') : 1.0
('become', 'one', 'of') : 1.0
('one', 'of', 'the') : 1.0
('the', 'fastest', 'growing') : 1.0
('fastest', 'growing', 'areas') : 1.0
('growing', 'areas', 'of') : 1.0
('areas', 'of', 'computer') : 1.0
('of', 'computer', 'science') : 1.0
('researchers', 'across', 'the') : 1.0
('acro

In [7]:

# ======================================================
# TASK 4
# Predict Top 5 Next Words
# ======================================================

def predict_bigram(word):

    candidates = {}

    for (w1,w2), count in bigrams.items():
        if w1 == word:
            candidates[w2] = count

    if len(candidates)==0:
        return []

    return sorted(
        candidates.items(),
        key=lambda x:x[1],
        reverse=True
    )[:5]


def predict_trigram(word1,word2):

    candidates = {}

    for (w1,w2,w3), count in trigrams.items():

        if w1==word1 and w2==word2:
            candidates[w3]=count

    if len(candidates)==0:
        return []

    return sorted(
        candidates.items(),
        key=lambda x:x[1],
        reverse=True
    )[:5]

print("\n" + "="*60)
print("TASK 4 : NEXT WORD PREDICTION")
print("="*60)

print("\nTop 5 after 'artificial'")

for word,count in predict_bigram("artificial"):
    print(word,"-",count)

print("\nTop 5 after 'natural language'")

for word,count in predict_trigram("natural","language"):
    print(word,"-",count)



TASK 4 : NEXT WORD PREDICTION

Top 5 after 'artificial'
intelligence - 6

Top 5 after 'natural language'
processing - 4


In [8]:

# ======================================================
# TASK 5
# Sentence Generation
# ======================================================

def generate_bigram(start,length=12):

    sentence=[start]

    while len(sentence)<length:

        choices=[]

        for (w1,w2),count in bigrams.items():
            if w1==sentence[-1]:
                choices.extend([w2]*count)

        if len(choices)==0:
            break

        sentence.append(random.choice(choices))

    return " ".join(sentence)


def generate_trigram(word1,word2,length=12):

    sentence=[word1,word2]

    while len(sentence)<length:

        choices=[]

        for (w1,w2,w3),count in trigrams.items():

            if w1==sentence[-2] and w2==sentence[-1]:
                choices.extend([w3]*count)

        if len(choices)==0:
            break

        sentence.append(random.choice(choices))

    return " ".join(sentence)

print("\n" + "="*60)
print("TASK 5 : GENERATED SENTENCES")
print("="*60)

print("\nBigram Sentence")
print(generate_bigram("artificial"))

print("\nTrigram Sentence")
print(generate_trigram("natural","language"))



TASK 5 : GENERATED SENTENCES

Bigram Sentence
artificial intelligence aims to unseen word sequences

Trigram Sentence
natural language processing research


In [9]:

# ======================================================
# TASK 6
# Model Comparison
# ======================================================

test_sentence = "Machine learning improves artificial intelligence applications"

words = tokenize(test_sentence)


In [10]:

# ---------- Bigram Probability ----------

bigram_prob = 1

for i in range(len(words)-1):

    pair=(words[i],words[i+1])

    if pair in bigram_probability:
        bigram_prob *= bigram_probability[pair]
    else:
        bigram_prob=0
        break


In [11]:

# ---------- Trigram Probability ----------

trigram_prob = 1

for i in range(len(words)-2):

    tri=(words[i],words[i+1],words[i+2])

    if tri in trigram_probability:
        trigram_prob *= trigram_probability[tri]
    else:
        trigram_prob=0
        break

print("\n" + "="*60)
print("TASK 6 : MODEL COMPARISON")
print("="*60)

print("Sentence:")
print(test_sentence)

print("\nBigram Probability :", bigram_prob)
print("Trigram Probability:", trigram_prob)

if trigram_prob > bigram_prob:
    print("\nTrigram Model performs better.")
elif bigram_prob > trigram_prob:
    print("\nBigram Model performs better.")
else:
    print("\nBoth models assign the same probability.")


TASK 6 : MODEL COMPARISON
Sentence:
Machine learning improves artificial intelligence applications

Bigram Probability : 0
Trigram Probability: 0

Both models assign the same probability.
